# Cell 1: Setup

In [4]:

import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import (
    load_faces,
    split_train_test,
    get_person_images,
    dataset_summary,
    print_summary
)
from src.visualizer import (
    plot_sample_faces,
    plot_single_face,
    plot_pixel_distribution,
    plot_person_grid,
    plot_matrix_info
)
from config import FIGURES_DIR

# مسیر ذخیره نمودارهای این نوت‌بوک
FIG_DIR = os.path.join(FIGURES_DIR, "01_data")

print("Setup complete.")

ModuleNotFoundError: No module named 'numpy'

# Cell 2: Load Data

In [2]:

X, y = load_faces(shuffle=True, random_state=42)

print(f"\nMatrix X dimensions: {X.shape}")
print(f"  Rows (m) = {X.shape[0]}  → number of images")
print(f"  Cols (n) = {X.shape[1]}  → pixels per image (64×64)")
print(f"\nLabel vector y: {y.shape}")
print(f"  Unique persons: {np.unique(y)}")

NameError: name 'load_faces' is not defined

# Cell 3: Dataset Summary

In [ ]:

summary = dataset_summary(X, y)
print_summary(summary)

# Cell 4: Visualize Sample Faces

In [ ]:

fig = plot_sample_faces(
    X, y,
    n_rows=4, n_cols=10,
    title="Sample Faces — Olivetti Dataset\n"
          "400 images from 40 persons (64×64 pixels each)",
    save_path=os.path.join(FIG_DIR, "01_sample_faces.png"),
    show=True
)

# Cell 5: All Images of Selected Persons

In [ ]:

# نمایش تمام ۱۰ تصویر از ۴ نفر انتخاب‌شده
selected_persons = [0, 1, 2, 3]

fig = plot_person_grid(
    X, y,
    person_ids=selected_persons,
    save_path=os.path.join(FIG_DIR, "02_person_grid.png"),
    show=True
)

print("توجه: هر فرد ۱۰ تصویر با زوایا، نورپردازی،")
print("و حالت‌های مختلف دارد.")

# Cell 6: Pixel Distribution

In [ ]:

fig = plot_pixel_distribution(
    X,
    save_path=os.path.join(FIG_DIR, "03_pixel_distribution.png"),
    show=True
)

# Cell 7: Matrix Information

In [ ]:

fig = plot_matrix_info(
    X,
    title="Data Matrix X ∈ ℝ^{400×4096}",
    save_path=os.path.join(FIG_DIR, "04_matrix_info.png"),
    show=True
)

# Cell 8: Vector Space Analysis

In [ ]:


print("=" * 55)
print("VECTOR SPACE ANALYSIS")
print("=" * 55)

m, n = X.shape
print(f"\n1. Data matrix X has shape: ({m}, {n})")
print(f"   Each row lives in ℝ^{n}")
print(f"   The ambient space is {n}-dimensional")

print(f"\n2. Maximum possible rank of X:")
print(f"   rank(X) ≤ min(m, n) = min({m}, {n}) = {min(m,n)}")
print(f"   → Our 400 images span a SUBSPACE of ℝ^{n}")
print(f"   → The subspace has dimension AT MOST {min(m,n)}")
print(f"   → That's {min(m,n)/n*100:.1f}% of the full space!")

print(f"\n3. Null space of X (as a linear map ℝ^n → ℝ^m):")
print(f"   dim(Null(X)) ≥ n - rank(X)")
print(f"   dim(Null(X)) ≥ {n} - {min(m,n)} = {n - min(m,n)}")
print(f"   → At least {n - min(m,n)} directions in ℝ^{n}")
print(f"     are NOT captured by our data!")

print(f"\n4. Key question:")
print(f"   Do all {min(m,n)} dimensions matter equally?")
print(f"   → Answer in Phase 3 (Spectral Analysis)")
print("=" * 55)

# Cell 9: Linear Independence Check

In [ ]:


# (We use SVD — the most stable method)
_, singular_values, _ = np.linalg.svd(X, full_matrices=False)

threshold     = 1e-10 * singular_values[0]  
n_significant = np.sum(singular_values > threshold)

print(f"Singular values (first 10): "
      f"{singular_values[:10].round(2)}")
print(f"Singular values (last 10):  "
      f"{singular_values[-10:].round(6)}")
print(f"\nNumerically significant singular values: {n_significant}")
print(f"Theoretical maximum: {min(m, n)}")
print(f"\nConclusion: The data spans a "
      f"{n_significant}-dimensional subspace of ℝ^{n}")


fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(singular_values, 'o-',
            color='steelblue', markersize=3,
            linewidth=1.5, alpha=0.8)
ax.set_xlabel("Index", fontsize=12)
ax.set_ylabel("Singular Value (log scale)", fontsize=12)
ax.set_title("Singular Values of X — Preview\n"
             "(Full spectral analysis in Phase 4)",
             fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "05_singular_values_preview.png"),
    dpi=150, bbox_inches="tight"
)
plt.show()

# Cell 10: Save Results

In [ ]:

import json

results = {
    "n_samples"              : int(m),
    "n_features"             : int(n),
    "n_persons"              : int(len(np.unique(y))),
    "max_theoretical_rank"   : int(min(m, n)),
    "numerical_rank_X"       : int(n_significant),
    "largest_singular_value" : float(singular_values[0]),
    "smallest_singular_value": float(singular_values[-1]),
    "ambient_space_dim"      : int(n),
    "subspace_dim_fraction"  : float(n_significant / n)
}

os.makedirs(
    os.path.join(os.path.dirname(os.getcwd()), "outputs", "results"),
    exist_ok=True
)

results_path = os.path.join(
    os.path.dirname(os.getcwd()),
    "outputs", "results", "01_data_analysis.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results saved.")
print(json.dumps(results, indent=2))